# Nivel Esencial - Gestión del Dataset y Limpieza de Texto

Este notebook cubre la **parte esencial** del proyecto:
1. Activar el entorno virtual
2. Descargar y ubicar el dataset
3. Carga inicial y exploración de los datos
4. Limpieza de texto con expresiones regulares
5. Eliminación de stopwords
6. Stemming y lematización

> Dataset: comentarios de YouTube etiquetados como odio (1) / no odio (0).

## 1. Activar el Entorno Virtual

Antes de ejecutar cualquier celda, debes activar el entorno virtual desde la **terminal** del proyecto.

**Windows (CMD / PowerShell):**
```
venv\Scripts\activate
```

**Mac / Linux:**
```
source venv/bin/activate
```

Una vez activado verás `(venv)` al inicio del prompt.  
Luego abre Jupyter con:
```
jupyter notebook
```

> En VS Code puedes seleccionar el intérprete directamente: `Ctrl+Shift+P` → "Python: Select Interpreter" → elige el de `venv`.

## 2. Descargar y Gestionar el Dataset

### Pasos para obtener el dataset:

1. Descarga el archivo desde este enlace:  
   [Youtube Comments Dataset](https://drive.google.com/file/d/1bG7fA273jIBgJfc6YS1vsKfr1qRiNUTU/view?usp=sharing)

2. Guarda el archivo descargado en la carpeta:
   ```
   data/raw/
   ```

3. La estructura esperada del CSV suele tener al menos dos columnas:
   - `comment_text` (o similar): el texto del comentario
   - `label` (o similar): 1 = odio, 0 = no odio

> Si el archivo está comprimido (.zip), descomprímelo dentro de `data/raw/` antes de continuar.

## 3. Carga Inicial de Datos

Cargamos el dataset y hacemos una exploración básica para entender su estructura antes de limpiarlo.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# --- Carga del dataset ---
FILE_PATH = "../data/raw/youtoxic_english_1000.csv"
df = pd.read_csv(FILE_PATH)

print(f"Dataset cargado: youtoxic_english_1000.csv")
print(f"Forma del dataset: {df.shape}")
print(f"Columnas: {df.columns.tolist()}")
df.head(3)

In [ ]:
# --- Exploración básica ---
print("Columnas:", df.columns.tolist())
print("\nTipos de datos:")
print(df.dtypes)
print("\nValores nulos por columna:")
print(df.isnull().sum())

# --- Columnas reales del dataset ---
TEXT_COL = "Text"
LABEL_COL = "IsToxic"   # etiqueta principal: True = tóxico, False = no tóxico

# Convertir la etiqueta a int (True → 1, False → 0)
df[LABEL_COL] = df[LABEL_COL].astype(int)

print(f"\nColumna de texto detectada: '{TEXT_COL}'")
print(f"Columna de etiqueta detectada: '{LABEL_COL}'")
print("\nDistribución de clases:")
print(df[LABEL_COL].value_counts())
print(f"\nPorcentaje de comentarios tóxicos: {df[LABEL_COL].mean()*100:.1f}%")

# Etiquetas disponibles (para referencia futura)
label_cols = ["IsToxic","IsAbusive","IsThreat","IsProvocative","IsObscene",
              "IsHatespeech","IsRacist","IsNationalist","IsSexist","IsHomophobic",
              "IsReligiousHate","IsRadicalism"]
print("\nEtiquetas disponibles:", label_cols)

# Visualización de la distribución de clases
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Distribución principal
df[LABEL_COL].value_counts().rename({0: "No tóxico", 1: "Tóxico"}).plot(
    kind="bar", ax=axes[0], color=["steelblue", "tomato"])
axes[0].set_title("Distribución: Tóxico vs No tóxico")
axes[0].set_xlabel("")
axes[0].set_ylabel("Cantidad")
axes[0].tick_params(rotation=0)

# Proporción de cada tipo de odio
toxicity_counts = df[label_cols].astype(int).sum().sort_values(ascending=False)
toxicity_counts.plot(kind="bar", ax=axes[1], color="coral")
axes[1].set_title("Frecuencia por tipo de contenido tóxico")
axes[1].set_xlabel("")
axes[1].set_ylabel("Cantidad")
axes[1].tick_params(rotation=45)

plt.tight_layout()
plt.show()

## 4. Limpieza de Texto con Expresiones Regulares

### ¿Por qué limpiar el texto?
El texto en bruto de YouTube contiene **ruido** que no aporta información al modelo:
- URLs (`https://...`)
- Menciones (`@usuario`)
- Emojis y caracteres especiales
- Números sueltos
- Espacios extra

Usamos **expresiones regulares (regex)** para eliminar todo ese ruido de forma eficiente.

### ¿Qué es regex?
Una expresión regular es un **patrón de búsqueda** que se aplica sobre texto. Ejemplos:
- `http\S+` → detecta cualquier URL
- `@\w+` → detecta menciones como `@usuario`
- `[^a-zA-Z\s]` → detecta todo lo que NO sea letra o espacio

In [ ]:
import re

def clean_text(text):
    """
    Limpia el texto aplicando expresiones regulares paso a paso.
    Dataset en inglés: youtoxic_english_1000.csv
    """
    # Paso 1: Convertir a minúsculas
    text = str(text).lower()

    # Paso 2: Eliminar URLs
    text = re.sub(r"http\S+|www\S+", "", text)

    # Paso 3: Eliminar menciones (@usuario)
    text = re.sub(r"@\w+", "", text)

    # Paso 4: Eliminar hashtags (#palabra)
    text = re.sub(r"#\w+", "", text)

    # Paso 5: Eliminar saltos de línea y tabulaciones
    text = re.sub(r"[\n\r\t]", " ", text)

    # Paso 6: Eliminar números
    text = re.sub(r"\d+", "", text)

    # Paso 7: Eliminar caracteres especiales (solo letras y espacios)
    text = re.sub(r"[^a-z\s]", "", text)

    # Paso 8: Eliminar espacios extra
    text = re.sub(r"\s+", " ", text).strip()

    return text

# Aplicar la función al dataset
df["text_clean"] = df[TEXT_COL].apply(clean_text)

# Comparar original vs limpio
print("=== Ejemplos antes y después de la limpieza ===\n")
for i in range(3):
    print(f"ORIGINAL: {df[TEXT_COL].iloc[i][:120]}...")
    print(f"LIMPIO:   {df['text_clean'].iloc[i][:120]}...")
    print("-" * 80)

# Longitud del texto antes y después
df["len_original"] = df[TEXT_COL].astype(str).apply(len)
df["len_clean"] = df["text_clean"].apply(len)
print(f"\nLongitud media original: {df['len_original'].mean():.0f} caracteres")
print(f"Longitud media limpia:   {df['len_clean'].mean():.0f} caracteres")

## 5. Eliminación de Stopwords

### ¿Qué son las stopwords?
Las **stopwords** son palabras muy frecuentes que no aportan significado al texto:  
`"el"`, `"la"`, `"de"`, `"que"`, `"y"`, `"en"`, `"a"` ...

Al eliminarlas, el modelo se concentra en las palabras **realmente importantes** para clasificar el odio.

### ¿Cómo funciona?
NLTK tiene listas predefinidas de stopwords en distintos idiomas. Las cargamos y filtramos el texto.

In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Descargar recursos de NLTK (solo la primera vez)
nltk.download("stopwords")
nltk.download("punkt")
nltk.download("punkt_tab")

# Dataset en inglés → solo stopwords en inglés
stop_words = set(stopwords.words("english"))
print(f"Total stopwords en inglés: {len(stop_words)}")
print("Ejemplos:", sorted(list(stop_words))[:15])

def remove_stopwords(text):
    """
    Tokeniza el texto y elimina las stopwords.
    Tokenizar = dividir la frase en palabras individuales (tokens).
    """
    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in stop_words and len(w) > 1]
    return " ".join(tokens)

df["text_no_sw"] = df["text_clean"].apply(remove_stopwords)

# Ver resultado
print("\n=== Antes y después de eliminar stopwords ===\n")
for i in range(3):
    print(f"CON SW:    {df['text_clean'].iloc[i][:100]}...")
    print(f"SIN SW:    {df['text_no_sw'].iloc[i][:100]}...")
    print("-" * 80)

## 6. Stemming y Lematización

### ¿Qué son y en qué se diferencian?

| Técnica | Descripción | Ejemplo |
|---------|-------------|---------|
| **Stemming** | Corta la palabra hasta su raíz (puede no ser una palabra real) | `"corriendo"` → `"corr"` |
| **Lematización** | Reduce la palabra a su forma base real (lema) | `"corriendo"` → `"correr"` |

**Cuándo usar cada una:**
- **Stemming**: más rápido, suficiente para muchos modelos clásicos (TF-IDF + Naive Bayes, SVM)
- **Lematización**: más preciso, mejor cuando el significado importa

Para este proyecto usaremos **stemming** (más eficiente para el nivel esencial).

In [ ]:
from nltk.stem import SnowballStemmer
from nltk.stem import WordNetLemmatizer

nltk.download("wordnet")

# Dataset en inglés
stemmer = SnowballStemmer("english")
lemmatizer = WordNetLemmatizer()

def apply_stemming(text):
    """Reduce cada palabra a su raíz. Ej: 'killing' → 'kill'"""
    tokens = text.split()
    return " ".join([stemmer.stem(w) for w in tokens])

def apply_lemmatization(text):
    """Reduce cada palabra a su forma base real. Ej: 'hating' → 'hate'"""
    tokens = text.split()
    return " ".join([lemmatizer.lemmatize(w) for w in tokens])

df["text_stemmed"] = df["text_no_sw"].apply(apply_stemming)
df["text_lemmatized"] = df["text_no_sw"].apply(apply_lemmatization)

# Ver comparativa completa del pipeline
print("=== Pipeline de preprocesamiento completo ===\n")
for i in range(2):
    print(f"ORIGINAL:   {df[TEXT_COL].iloc[i][:90]}...")
    print(f"LIMPIO:     {df['text_clean'].iloc[i][:90]}...")
    print(f"SIN SW:     {df['text_no_sw'].iloc[i][:90]}...")
    print(f"STEMMING:   {df['text_stemmed'].iloc[i][:90]}...")
    print(f"LEMATIZADO: {df['text_lemmatized'].iloc[i][:90]}...")
    print("=" * 80)

In [ ]:
## Guardar el dataset preprocesado

# Conservamos texto original, etiqueta IsToxic y el texto procesado listo para el modelo
df_processed = df[[TEXT_COL, LABEL_COL, "text_clean", "text_no_sw", "text_lemmatized"]].copy()
df_processed.rename(columns={"text_lemmatized": "text_processed"}, inplace=True)

# Eliminar filas vacías tras la limpieza
df_processed = df_processed[df_processed["text_processed"].str.strip() != ""]
df_processed.dropna(subset=["text_processed"], inplace=True)

print(f"Filas originales:       {len(df)}")
print(f"Filas tras limpieza:    {len(df_processed)}")
print(f"\nDistribución final de '{LABEL_COL}':")
print(df_processed[LABEL_COL].value_counts())
print(f"Tóxicos: {df_processed[LABEL_COL].sum()} | No tóxicos: {(df_processed[LABEL_COL]==0).sum()}")

# Guardar
OUTPUT_PATH = "../data/processed/dataset_procesado.csv"
df_processed.to_csv(OUTPUT_PATH, index=False)
print(f"\nDataset procesado guardado en: {OUTPUT_PATH}")
df_processed.head()